# Problem 4

In [ ]:
import numpy as np
import pandas as pd
from cmdstanpy import CmdStanModel

predictors = ['attr', 'shar', 'intel']
seed = 42

df = pd.read_csv('speed_data_data.csv')[['dec', *predictors]].dropna()
df['dec'] = df['dec'].astype(int)

rng = np.random.default_rng(seed)
order = rng.permutation(len(df))
split = int(0.8 * len(df))

train = df.iloc[order[:split]]
test = df.iloc[order[split:]]

means = train[predictors].mean()
sds = train[predictors].std(ddof=0)

x_train = ((train[predictors] - means) / sds).to_numpy()
x_test = ((test[predictors] - means) / sds).to_numpy()
y_train = train['dec'].to_numpy(dtype=int)
y_test = test['dec'].to_numpy(dtype=int)

stan_data = {
    'N': len(train),
    'P': len(predictors),
    'x': x_train,
    'y': y_train.tolist(),
    'N_test': len(test),
    'x_test': x_test,
}

model = CmdStanModel(stan_file='speed_dating.stan')
fit = model.sample(
    data=stan_data,
    seed=seed,
    chains=4,
    parallel_chains=4,
    iter_warmup=500,
    iter_sampling=500,
    show_progress=False,
)

p_test_draws = fit.stan_variable('p_test')
p_mean = p_test_draws.mean(axis=0)
y_hat = (p_mean >= 0.5).astype(int)

brier = float(np.mean((p_mean - y_test) ** 2))
accuracy = float(np.mean(y_hat == y_test))

majority_class = int(y_train.mean() >= 0.5)
baseline_accuracy = float(np.mean(np.full_like(y_test, majority_class) == y_test))
baseline_brier = float(np.mean((np.full_like(y_test, majority_class, dtype=float) - y_test) ** 2))

brier_by_draw = np.mean((p_test_draws - y_test) ** 2, axis=1)
accuracy_by_draw = np.mean((p_test_draws >= 0.5) == y_test, axis=1)

def ci(values):
    return [float(np.quantile(values, 0.025)), float(np.quantile(values, 0.975))]

print('model Brier score:', brier)
print('model accuracy:', accuracy)
print('baseline Brier score:', baseline_brier)
print('baseline accuracy:', baseline_accuracy)
print('draw-by-draw Brier:', float(brier_by_draw.mean()), ci(brier_by_draw))
print('draw-by-draw accuracy:', float(accuracy_by_draw.mean()), ci(accuracy_by_draw))


c:\Users\Victor\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
23:37:32 - cmdstanpy - INFO - CmdStan start processing
23:37:32 - cmdstanpy - INFO - Chain [1] start processing
23:37:32 - cmdstanpy - INFO - Chain [2] start processing
23:37:32 - cmdstanpy - INFO - Chain [3] start processing
23:37:32 - cmdstanpy - INFO - Chain [4] start processing
23:37:34 - cmdstanpy - INFO - Chain [1] done processing
23:37:34 - cmdstanpy - INFO - Chain [2] done processing
23:37:34 - cmdstanpy - INFO - Chain [3] done processing
23:37:34 - cmdstanpy - INFO - Chain [4] done processing


model Brier score: 0.1702628711457519
model accuracy: 0.7405368203716449
baseline Brier score: 0.428079834824501
baseline accuracy: 0.5719201651754989
draw-by-draw Brier: 0.17038535059892626 [0.16994187955354265, 0.1710075656283676]
draw-by-draw accuracy: 0.7437680660701995 [0.7405368203716449, 0.7501720578114246]


Answer:

The model Brier score was 0.170 and the model accuracy was 0.741. The baseline Brier score was 0.428 and the baseline accuracy was 0.572, so the model did better than the baseline.

For this model, the draw-by-draw Brier mean was 0.170 with a 95% interval of [0.1699, 0.1710]. The draw-by-draw accuracy mean was 0.744 with a 95% interval of [0.741, 0.750].